In [8]:
import numpy as np
import pandas as pd
import  tpqoa
import time
from datetime import datetime, timedelta, timezone
import warnings
warnings.filterwarnings('ignore')

In [17]:
class ConTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, window, units):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.data = None
        self.last_bar = None
        self.units = units
        self.position = 0
        self.profits = [] # NEW

        #*****************add strategy-specific attributes here******************
        self.window = window
        #************************************************************************

    def get_most_recent(self, days = 5):
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        now = now - timedelta(microseconds = now.microsecond)
        past = now - timedelta(days = days)
        df = self.get_history(instrument = self.instrument, start = past, end = now,
                               granularity = "S5", price = "M", localize = False).c.dropna().to_frame()
        df.rename(columns = {"c":self.instrument}, inplace = True)
        df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]
        self.raw_data = df.copy()
        self.last_bar = self.raw_data.index[-1]


    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")


        # collect and store tick data
        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        # if a time longer than the bar_lenght has elapsed between last full bar and the most recent tick
        if recent_tick - self.last_bar >= self.bar_length:
            self.resample_and_join()
            self.moving_average()
            self.execute_trades()

    def resample_and_join(self):
        self.raw_data = pd.concat([self.raw_data, self.tick_data.resample(self.bar_length,
                                                                          label="right").last().ffill().iloc[:-1]])
        self.tick_data = self.tick_data.iloc[-1:]
        self.last_bar = self.raw_data.index[-1]

    # def define_strategy(self): # "strategy-specific"
    #     df = self.raw_data.copy()
    #
    #     #******************** define your strategy here ************************
    #     df["returns"] = np.log(df[self.instrument] / df[self.instrument].shift())
    #     df["position"] = -np.sign(df.returns.rolling(self.window).mean())
    #     #***********************************************************************
    #
    #     self.data = df.copy()

    def moving_average(self):
        df = self.raw_data.copy()
        sma_l = 20
        sma_h = 50
        df['SMA_20'] = df[self.instrument].rolling(20).mean()
        df['SMA_50'] = df[self.instrument].rolling(50).mean()
        df['direction'] = np.where(df['SMA_20'] > df['SMA_50'], 1, -1)

        self.data = df.copy()

    def execute_trades(self):
        if self.data["position"].iloc[-1] == 1:
            if self.position == 0:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            elif self.position == -1:
                order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            self.position = 1
        elif self.data["position"].iloc[-1] == -1:
            if self.position == 0:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            self.position = -1
        elif self.data["position"].iloc[-1] == 0:
            if self.position == -1:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            self.position = 0

    def report_trade(self, order, going):  # NEW
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print("{} | {}".format(time, going))
        print("{} | units = {} | price = {} | P&L = {} | Cum P&L = {}".format(time, units, price, pl, cumpl))
        print(100 * "-" + "\n")


In [18]:
trader = ConTrader('oanda.cfg', 'EUR_USD', bar_length='1min', window=60, units=1000000)

In [ ]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop= 200)

if trader.position!= 1:
    close_order= trader.create_order(trader.instrument, units=-trader.position * trader.units, suppress=True, ret=True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position=0

In [23]:
trader.tick_data

,EUR_USD
2026-02-20 21:59:07.538256848+00:00,1.178165


In [29]:
trader.on_success('2026-02-20 23:20:00', 1.19055, 1.19065)

1 

TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects.

In [20]:
api = tpqoa.tpqoa('oanda.cfg')

In [24]:
len(api.get_instruments())

127